# 02 - ICB boundaries

Runs in Google Colab. Downloads the full-resolution boundaries of the 42 Integrated Care Boards (Integrated Care Boards (April 2023) EN BFE, ONS Open Geography Portal) to Google Drive as `icb_boundaries.geojson`. Copy that file into `data/` and run `python code/simplify_boundaries.py`, which makes the simplified `data/icb_boundaries_small.geojson` used by the analysis. Steps 2 and 3 are optional preview maps from the output of notebook 01.

Source: Office for National Statistics licensed under the Open Government Licence v.3.0. Contains OS data © Crown copyright and database right 2023.

In [ ]:
!pip -q install geopandas

In [ ]:
# 1. Mount Drive and fetch the ICB (April 2023) boundaries
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive'
import requests, geopandas as gpd
URL = ('https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/'
       'Integrated_Care_Boards_April_2023_EN_BFE/FeatureServer/0/query'
       '?outFields=ICB23CD,ICB23NM&where=1%3D1&f=geojson')
open(f'{DRIVE_DIR}/icb_boundaries.geojson', 'w').write(requests.get(URL, timeout=180).text)
gdf = gpd.read_file(f'{DRIVE_DIR}/icb_boundaries.geojson')
print('boundaries saved to', DRIVE_DIR, gdf.shape)

In [ ]:
# 2. (Optional) 42-area shares of each item from the output of notebook 01
import pandas as pd, numpy as np, re
FULL = pd.read_csv(f'{DRIVE_DIR}/icb_area_substance_items_full.csv')
area_drug = FULL.groupby(['AREA_CODE','DRUG'], as_index=False)['ITEMS'].sum()
area_drug['share'] = area_drug['ITEMS'] / area_drug.groupby('AREA_CODE')['ITEMS'].transform('sum')
icbname = (FULL[FULL['AREA_NAME'].str.contains('INTEGRATED CARE', case=False, na=False)]
           .groupby('AREA_CODE')['AREA_NAME'].first())
def norm(s): return re.sub('[^a-z]', '', str(s).lower().replace('nhs','').replace('integrated care board',''))
gdf['key'] = gdf['ICB23NM'].map(norm)
code2key = {c: norm(n) for c, n in icbname.items()}
print('areas matched to boundary:', sum(v in set(gdf['key']) for v in code2key.values()), '/', len(code2key))

In [ ]:
# 3. (Optional) preview maps of the within-area share of six example items
import matplotlib.pyplot as plt
EXAMPLES = ['Thyrotropin alfa','Chlortalidone','Cladribine','Dapagliflozin','Ketoprofen','Catheters']
fig, axes = plt.subplots(2, 3, figsize=(15, 12))
for ax, drug in zip(axes.flat, EXAMPLES):
    sub = area_drug[area_drug['DRUG'].str.fullmatch(drug, case=False, na=False)]
    if not len(sub): sub = area_drug[area_drug['DRUG'].str.contains(drug, case=False, na=False)]
    sub = sub.groupby('AREA_CODE', as_index=False)['share'].sum()
    sub['key'] = sub['AREA_CODE'].map(code2key)
    m = gdf.merge(sub, on='key', how='left')
    m.plot(column='share', cmap='OrRd', legend=True, ax=ax, edgecolor='0.6', linewidth=0.3,
           missing_kwds={'color':'#eeeeee'})
    ax.set_title(drug, fontweight='bold'); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/icb_drug_maps.png', dpi=200, bbox_inches='tight'); plt.show()

**Done.** `icb_boundaries.geojson` is in `DRIVE_DIR`. The file used for the paper has SHA-256 86b8ee43371839c2af75be45bf2418d867ef10831d1a855c79aa8de3877c422a; ONS may revise the boundaries. If the service address above stops working, the same dataset (Integrated Care Boards (April 2023) EN BFE) can be downloaded as GeoJSON from the ONS Open Geography Portal, https://geoportal.statistics.gov.uk.